In [ ]:
from ngsxditto import *
from ngsolve import *
from xfem import *
import ngsolve.webgui as ngw
from netgen.occ import *

In [ ]:
domain = MoveTo(-1, -1).Rectangle(2, 2).Face()
domain.edges.Max(X).name = "right"
domain.edges.Min(X).name = "left"
domain.edges.Min(Y).name = "bottom"
domain.edges.Max(Y).name = "top"
mesh = Mesh(OCCGeometry(domain, dim=2).GenerateMesh(maxh=0.15))

In [ ]:
dt = 0.05
t = Parameter(0)
starting_levelset = (5*x**2 + y**2)**(1/2) - 2.0/3.0
transport = ExplicitDGTransport(mesh, dt=dt, order=2, compile=False)
levelset = LevelSetGeometry(transport)
levelset.Initialize(starting_levelset)

In [ ]:
fluid1_params = FluidParameters(viscosity=1)
fluid2_params = FluidParameters(viscosity=1e-3)

mean_curvature = MeanCurvatureSolver(mesh, order=2, lset=levelset)
mean_curvature.Step()
fluid = TwoPhaseTaylorHood(mesh, fluid1_params=fluid1_params, fluid2_params=fluid2_params, lset=levelset, surface_tension=mean_curvature.H, dt=dt, order=3, ghost_stab=1)
fluid.Initialize(dirichlet={".*": CF((0, 0))})
fluid.ValidateStep()
sol = fluid.SolveStokes()
u1, p1, u2, p2 = sol.components
fluid.SetInitialValues(u1, u2, p1, p2)
ngw.Draw(IfPos(levelset.lsetp1, fluid.gfu.components[2], fluid.gfu.components[0]), mesh, "u", deformation=levelset.deformation)